# Lesson 01 - AI Agent မိတ္ဆက္

**AI Agents for Beginners** သင္တန္းရဲ႕ ပထမ ဆုိဒ္သို႔ မဂၤလာပါ။

**AI agent** ဆိုတာက ပိုင္းစိတ္ေက်ြးေက်ြးၿပီး အသံုးျပဳသူအၾကံျပဳခ်က္အတြက္ ရည္မွန္းခ်က္တစ္ခုကို လုပ္ေဆာင္ဖို႔ အေျခခံအျဖစ္ အႀကီးစားေျမာက္ေသာ ဘာသာစကားမော်ဒယ် (LLM) ကို အျကြမ္းထဲသံုးၿပီး အလုပ္လုပ္ႏိုင္ေသာ ပ႐ုိဂရမ္တစ္ခုျဖစ္ပါတယ္။ ဥပမာ - API ေခၚဆိုျခင္း၊ ဒေတာေၾကာင္းမ်ားအား ေမးသံုးျခင္း၊ ေကာ္ဒ္ေရးနုိင္ျခင္း စသည္ျဖင့္ လက္ေတြ႔ေလာကမွာ လုပ္ေဆာင္ႏိုင္ပါတယ္။

ဤ notebook အတြင္းသင္သည္ သင့္၏ ပထမဆုံး agent တစ္ခုျဖစ္ေသာ **ခရီးသြားအေၾကာင္းAgent** ကို တည္ေဆာက္မည္ ျဖစ္ၿပီး ထိုကဲ့သို႔ သင္သည္ ေအာက္ပါအခ်က္မ်ားကို ေလ့လာပါမည္-

1. **Microsoft Agent Framework** ကို သံုးၿပီး Azure AI Foundry Agent Service ႏွင့္ ခ်ိတ္ဆက္နည္း။
2. အဲဒီ agent ကို ေခၚႏိုင္ေသာ မူလ Python function တစ္ခုျဖစ္သည့္ **ကိရိယာ** တစ္ခုေပးျခင္း။
3. Agent ကို ေဆာင္ရြက္ၿပီး ၎၏ တုံ႔ျပန္မႈကို စစ္ေဆးျခင္း။
4. Agent ၏ တုံ႔ျပန္မႈကို စကားလံုးတိုင္းစီ အစိတ္အပိုင္းအလိုက္ ျပန္လည္ႀကည့္ရႈျခင္း။


## Setup

ဒီ notebook ကို run မလုပ်ခင်မှာ အောက်ပါအချက်တွေကို သေချာစစ်ဆေးပါ။

1. **Chat model တစ်ခု (ဥပမာ - `gpt-4o-mini`) deploy လုပ်ထားတဲ့ Azure AI Foundry project**
2. **Azure CLI ဖြင့် login ပြီးခြင်း** — terminal မှာ `az login` ကို run ပါ။
3. **လိုအပ်တဲ့ environment variables များကို သတ်မှတ်ခြင်း:**
   - `AZURE_AI_PROJECT_ENDPOINT` — သင့် Azure AI Foundry project endpoint
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — သင့် deploy လုပ်ထားသော model အမည်

အောက်က cell မှာ သင်လိုအပ်တဲ့ Python package များကို install လုပ်ပေးပါသည်။


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## သင်၏ ပထမဆုံး ကိုယ်စားလှယ်ကို ဖန်တီးခြင်း

ကိုယ်စားလှယ်တစ်ယောက်အတွက် လိုအပ်တာ ၂ ခုပဲရှိတယ်-

- **လမ်းညွန်ချက်များ** ဟာ ၎င်းကို *ဘယ်သူလဲ*၊ *ဘယ်လိုလုပ်ရမလဲ* ဆိုတာပြောပြပေးတဲ့ အရာ (စနစ် prompt) ဖြစ်တယ်။
- **ကိရိယာများ** — ကိုယ်စားလှယ်က အချက်အလက်ရယူဖို့ သို့မဟုတ် လုပ်ဆောင်ချက်များပြုလုပ်ဖို့ ခေါ်ယူနိုင်တဲ့ `@tool` နဲ့ အလှူခံထားတဲ့ Python function များ။

အောက်မှာ လူကြိုက်များတဲ့ နွားရာရာသီ ခရီးစဉ်ရပ်ကွက်များစာရင်း ပြန်ပေးတဲ့ ရိုးရှင်းတဲ့ကိရိယာတစ်ခုကို သတ်မှတ်ထားပါတယ်။ အသုံးပြုသူက ခရီးသွားအကြံပြုချက်တွေ မေးလာရင် ကိုယ်စားလှယ်က ဒီကိရိယာကို အသုံးပြုပါလိမ့်မယ်။


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Responses

ပိုမိုအပြောင်းအလဲရှိတဲ့ အတွေ့အကြုံအတွက် agent ရဲ့ တုံ့ပြန်ချက်ကို **stream** ပေးနိုင်ပါတယ်။ ပြီးပြည့်စုံတဲ့ တုံ့ပြန်ချက်ကို စောင့်မနေဘဲ၊ agent က စာသား အပိုင်းအစတွေကို ထုတ်ပေးနေပါတယ်။ ဒါက chat အင်တာဖေ့စ်တွေမှာ အထူးအသုံးဝင်ပြီး output ကို တိုက်ရိုက် ပြသချင်တဲ့အခါမှာ အသုံးဝင်ပါတယ်။


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

ဒီစာသင်ခန်းမှာ သင်သိရှိခဲ့တာတွေက:

- **provider တစ်ခု ဖန်တီးခြင်း** ကို `AzureAIProjectAgentProvider` မှတဆင့် Azure AI Foundry Agent Service နှင့် ချိတ်ဆက်ဖို့။
- **tool တစ်ခု သတ်မှတ်ခြင်း** ကို `@tool` decorator အသုံးပြုပြီး agent သည် သင်၏ Python function များကို ခေါ်နိုင်စေရန်။
- **agent ကို run ပြုလုပ်ခြင်း** အတွက် အသုံးပြုသူသို့မက်ဆေ့ချ်ပို့ပြီး ၎င်း၏ တုံ့ပြန်ချက်ကို မျက်နှာပြင်ပေါ် ပြသခြင်း။
- **response များကို stream ဖြင့်** အချိန်နှင့်တပြေးညီ ထုတ်ပြန်ခြင်း။

နောက်ထပ် စာသင်ခန်းတွင် agent အခြေခံ framework များကို ပိုမိုနက်ရှိုင်းစွာလေ့လာပြီး agent များကို ပိုမိုအင်အားကြီးသော tool များနှင့် များပြားသော အဆင့်ကောက်တွ၍စဉ်းစားစနစ်များ ပေးဆောင်နည်းများကို သင်ယူမည်ဖြစ်သည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ခြောက်ချက်ချက်ချက်**  
ဤစာတမ်းကို AI ဘာသာပြန်ဆာဗစ် [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြုပြီး ဘာသာပြန်ခြေဟုဖြစ်သည်။ တိကျမှုအတွက်ကြိုးစားနေသော်လည်း အလိုအလျောက်ဘာသာပြန်မှုများတွင် မှားယွင်းမှုများ သို့မဟုတ် တိကျမှုမရှိမှုများပါဝင်နိုင်ကြောင်း သတိပြုရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မိခင်ဘာသာဖြင့်သာ တရားဝင်အထောက်အထားအဖြစ် ယူဆသင့်ပါသည်။ အရေးကြီးသည့် အချက်အလက်များအတွက် နိုင်ငံတော်အရည်အချင်းပြည့်မီသော လူသားဘာသာပြန်အဖွဲ့ကို အသုံးပြုရန် အကြံပြုပါသည်။ ဤဘာသာပြန်မှုကို အသုံးပြုခြင်းကြောင့် ဖြစ်ပေါ်ဖြစ်နိုင်သည့် ခွဲခြားမှားယွင်းမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
